# Dual Franka absolute EE pose smoke test

This notebook runs from the local Le-nero environment and talks to the ZeroRPC server on the robot machine. It tests `dual_robot_move_to_ee_pose(..., delta=False)` with a very small absolute target built from the current end-effector poses.

Pose format is `[x, y, z, rx, ry, rz]` in the server torso frame. Rotation is rotvec in radians.

## 1. Imports and repo path

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import numpy as np

REPO_ROOT = Path('/home/deepcybo/Le-nero/dual_arm_teleop')
if not (REPO_ROOT / 'robots' / 'dual_franka').exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'robots' / 'dual_franka').exists():
            REPO_ROOT = candidate
            break
sys.path.insert(0, str(REPO_ROOT))

from robots.dual_franka.dual_franka_robotiq_rpc_client import (
    DualFrankaRobotiqRpcClient,
    _absolute_target_to_delta,
    _plan_smooth_absolute_trajectory,
    _pose_from_side_observation,
)

np.set_printoptions(precision=6, suppress=True)
print('repo:', REPO_ROOT)

## 2. Connect to the remote server

In [ ]:
SERVER_HOST = os.environ.get('FRANKA_RPC_HOST', '172.16.0.1')
SERVER_PORT = int(os.environ.get('FRANKA_RPC_PORT', '4242'))
RPC_TIMEOUT_SEC = float(os.environ.get('FRANKA_RPC_TIMEOUT_SEC', '30'))

client = DualFrankaRobotiqRpcClient(
    ip=SERVER_HOST,
    port=SERVER_PORT,
    timeout=RPC_TIMEOUT_SEC,
)

print('server:', f'{SERVER_HOST}:{SERVER_PORT}')
print('ping:', client.ping())

## 3. Read current absolute EE poses

In [ ]:
def read_ee_poses():
    obs = client.get_observation()
    left = np.array(_pose_from_side_observation(obs, 'left_arm'), dtype=float)
    right = np.array(_pose_from_side_observation(obs, 'right_arm'), dtype=float)
    return obs, left, right

obs0, left0, right0 = read_ee_poses()
print('left0 :', np.array2string(left0, precision=6, suppress_small=True))
print('right0:', np.array2string(right0, precision=6, suppress_small=True))
print('observation keys:', list(obs0.keys()) if isinstance(obs0, dict) else type(obs0))

## 4. Build a tiny absolute target

In [ ]:
# Safe default: move both EEs 5 mm upward in torso-frame z, with orientation unchanged.
# Keep this smoke test near the current pose. Large absolute moves should be planned as waypoints.
LEFT_OFFSET = np.array([0.0, 0.0, 0.5, 0.0, 0.0, 0.0], dtype=float)
RIGHT_OFFSET = np.array([0.0, 0.0, 0.5, 0.0, 0.0, 0.0], dtype=float)

# Manual absolute targets are intentionally disabled by default.
# If you enable this, set a distinct left/right target near each arm's current pose.
USE_MANUAL_ABSOLUTE_TARGET = False
MANUAL_LEFT_TARGET = left0.copy()
MANUAL_RIGHT_TARGET = right0.copy()

# Guardrails for this notebook. These are deliberately conservative.
MAX_TOTAL_TRANSLATION_M = 2.0
MAX_TOTAL_ROTATION_RAD = 1.57
MIN_LEFT_RIGHT_TARGET_DISTANCE_M = 0.10
REQUIRE_LEFT_Y_POSITIVE_RIGHT_Y_NEGATIVE = True
ALLOW_LARGE_TARGET = False

def _pose_delta(current, target):
    return np.array(_absolute_target_to_delta(current.tolist(), target.tolist()), dtype=float)

def validate_absolute_targets(left_current, right_current, left_target, right_target):
    left_current = np.asarray(left_current, dtype=float).reshape(6)
    right_current = np.asarray(right_current, dtype=float).reshape(6)
    left_target = np.asarray(left_target, dtype=float).reshape(6)
    right_target = np.asarray(right_target, dtype=float).reshape(6)
    for name, pose in [('left_target', left_target), ('right_target', right_target)]:
        if not np.all(np.isfinite(pose)):
            raise ValueError(f'{name} contains NaN/Inf: {pose}')

    left_delta = _pose_delta(left_current, left_target)
    right_delta = _pose_delta(right_current, right_target)
    checks = [
        ('left translation', np.linalg.norm(left_delta[:3]), MAX_TOTAL_TRANSLATION_M),
        ('right translation', np.linalg.norm(right_delta[:3]), MAX_TOTAL_TRANSLATION_M),
        ('left rotation', np.linalg.norm(left_delta[3:]), MAX_TOTAL_ROTATION_RAD),
        ('right rotation', np.linalg.norm(right_delta[3:]), MAX_TOTAL_ROTATION_RAD),
    ]
    errors = []
    if not ALLOW_LARGE_TARGET:
        for label, value, limit in checks:
            if value > limit:
                errors.append(f'{label} delta {value:.4f} exceeds limit {limit:.4f}')

    target_distance = np.linalg.norm(left_target[:3] - right_target[:3])
    if target_distance < MIN_LEFT_RIGHT_TARGET_DISTANCE_M:
        errors.append(
            f'left/right targets are only {target_distance:.4f} m apart; this often means both arms were given the same target'
        )

    if REQUIRE_LEFT_Y_POSITIVE_RIGHT_Y_NEGATIVE and (left_target[1] <= 0.0 or right_target[1] >= 0.0):
        errors.append(
            'target y signs look swapped/crossed for this setup: expected left y > 0 and right y < 0'
        )

    print('left target :', np.array2string(left_target, precision=6, suppress_small=True))
    print('right target:', np.array2string(right_target, precision=6, suppress_small=True))
    print('left delta  :', np.array2string(left_delta, precision=6, suppress_small=True))
    print('right delta :', np.array2string(right_delta, precision=6, suppress_small=True))
    print('left delta norm m/rad :', np.linalg.norm(left_delta[:3]), np.linalg.norm(left_delta[3:]))
    print('right delta norm m/rad:', np.linalg.norm(right_delta[:3]), np.linalg.norm(right_delta[3:]))

    if errors:
        raise ValueError('Unsafe absolute target for this smoke test:\n- ' + '\n- '.join(errors))
    return left_delta, right_delta

if USE_MANUAL_ABSOLUTE_TARGET:
    left_target = np.array(MANUAL_LEFT_TARGET, dtype=float)
    right_target = np.array(MANUAL_RIGHT_TARGET, dtype=float)
else:
    left_target = left0 + LEFT_OFFSET
    right_target = right0 + RIGHT_OFFSET

left_delta_preview, right_delta_preview = validate_absolute_targets(left0, right0, left_target, right_target)

## 5. Preview and send the smoothed absolute target

Leave `DRY_RUN=True` for the first pass. This cell re-reads the live pose, validates the target, previews the client-side S-curve trajectory, and only sends when `DRY_RUN=False`.

In [ ]:
DRY_RUN = True

left_target = np.array([0.7, 0.7, 0.2, 1.57, -1.57, 0.7], dtype=float)
right_target = np.array([0.7, -0.7, 0.2, -1.57, -1.57, -0.7], dtype=float)

TRAJECTORY_KWARGS = dict(
    duration_sec=None,             # None: derive duration from speed limits
    rate_hz=50.0,
    max_translation_speed=0.05,    # m/s; raise slowly only after accuracy/stability is good
    max_rotation_speed=0.30,       # rad/s
    max_translation_step=0.002,    # m per streamed waypoint
    max_rotation_step=0.015,       # rad per streamed waypoint
    min_duration_sec=0.8,
    max_steps=3000,
    settle_time_sec=0.8,           # wait after each streamed segment before measuring error
    position_tolerance_m=0.001,    # 1 mm target tolerance for correction loop
    rotation_tolerance_rad=0.008,  # about 0.46 deg
    max_correction_iters=2,
)

# Re-read just before planning so the absolute target is checked against the live pose.
_, left_start, right_start = read_ee_poses()
validate_absolute_targets(left_start, right_start, left_target, right_target)
preview_kwargs = {k: v for k, v in TRAJECTORY_KWARGS.items() if k not in {
    'settle_time_sec', 'position_tolerance_m', 'rotation_tolerance_rad', 'max_correction_iters'
}}
preview, preview_meta = _plan_smooth_absolute_trajectory(
    left_start.tolist(),
    right_start.tolist(),
    left_target.tolist(),
    right_target.tolist(),
    **preview_kwargs,
)
print('planned steps:', preview_meta['steps'])
print('planned duration_sec:', preview_meta['duration_sec'])
print('planned period_sec:', preview_meta['period_sec'])

if DRY_RUN:
    print('DRY_RUN=True, no robot command was sent.')
else:
    result = client.dual_robot_move_to_ee_pose(
        left_target.tolist(),
        right_target.tolist(),
        delta=False,
        wait=False,
        smooth=True,
        **TRAJECTORY_KWARGS,
    )
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str)[:6000])

## 6. Verify the resulting pose

In [ ]:
obs1, left1, right1 = read_ee_poses()
left_residual = np.array(_absolute_target_to_delta(left1.tolist(), left_target.tolist()), dtype=float)
right_residual = np.array(_absolute_target_to_delta(right1.tolist(), right_target.tolist()), dtype=float)

print('left1 :', np.array2string(left1, precision=6, suppress_small=True))
print('right1:', np.array2string(right1, precision=6, suppress_small=True))
print('left actual delta :', np.array2string(left1 - left0, precision=6, suppress_small=True))
print('right actual delta:', np.array2string(right1 - right0, precision=6, suppress_small=True))
print('left actual-target xyz/rotvec raw :', np.array2string(left1 - left_target, precision=6, suppress_small=True))
print('right actual-target xyz/rotvec raw:', np.array2string(right1 - right_target, precision=6, suppress_small=True))
print('left residual command to target :', np.array2string(left_residual, precision=6, suppress_small=True))
print('right residual command to target:', np.array2string(right_residual, precision=6, suppress_small=True))
print('left residual norm m/rad :', np.linalg.norm(left_residual[:3]), np.linalg.norm(left_residual[3:]))
print('right residual norm m/rad:', np.linalg.norm(right_residual[:3]), np.linalg.norm(right_residual[3:]))
if 'result' in globals():
    print('client ok:', result.get('ok'))
    print('client final_error:', json.dumps(result.get('final_error'), indent=2, ensure_ascii=False, default=str))

## 7. Optional: move back to the captured start pose

In [ ]:
MOVE_BACK = True

if MOVE_BACK:
    result = client.dual_robot_move_to_ee_pose(
        left0.tolist(),
        right0.tolist(),
        delta=False,
        wait=False,
    )
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))
    time.sleep(1.0)
    _, left_back, right_back = read_ee_poses()
    print('left back error :', np.array2string(left_back - left0, precision=6, suppress_small=True))
    print('right back error:', np.array2string(right_back - right0, precision=6, suppress_small=True))

## 8. Close the client

In [ ]:
client.close()